# Laboratorium 2 — świat Wumpusa i wnioskowanie w ASP (Clingo)

## Problem

*Świat Wumpusa* (Russell & Norvig, *Artificial Intelligence: A Modern Approach*, rozdz. 7) to klasyczny przykład środowiska dla **agenta opartego na wiedzy**. Jaskinia jest prostokątną planszą pól $(x, y)$, $1 \le x \le W$, $1 \le y \le H$. Myśliwy zaczyna na polu wejściowym $(1,1)$, patrząc na wschód. W jaskini znajdują się:

* **jamy** — wejście do jamy oznacza śmierć,
* **wumpus** — potwór, który zjada myśliwego wchodzącego na jego pole (martwy wumpus jest już niegroźny),
* **złoto** — cel wyprawy.

Pole wejściowe jest zawsze bezpieczne. Agent **nie widzi** planszy — zna tylko jej rozmiar oraz to, co poczuje na polach, które odwiedził:

| bodziec | znaczenie |
|---|---|
| *Stench* (smród) | na polu sąsiednim (krawędzią) jest wumpus — także martwy |
| *Breeze* (powiew) | na polu sąsiednim jest jama |
| *Glitter* (blask) | na bieżącym polu leży złoto |
| *Bump* (uderzenie) | agent wszedł w ścianę |
| *Scream* (krzyk) | właśnie wystrzelona strzała zabiła wumpusa |

**Akcje:** `left`, `right` (obrót o 90°), `forward` (krok), `grab` (podniesienie złota), `shoot` (strzała leci prosto do ściany i zabija pierwszego napotkanego żywego wumpusa), `climb` (wyjście z jaskini, tylko na polu $(1,1)$).

**Punktacja (jak w AIMA):** $+1000$ za każde wyniesione złoto, $-1000$ za śmierć, $-10$ za strzałę, $-1$ za każdą akcję.

## Podejście

Agent zapisuje w bazie wiedzy wyłącznie obserwacje (odwiedzone pola, percepcje, strzały). Reguły świata zapisane są w programie ASP `wumpus_kb.lp`: każdy model stabilny to jeden możliwy układ ukrytej części jaskini zgodny z obserwacjami. Clingo liczy

* **konsekwencje ostrożne** (prawdziwe we wszystkich modelach) — to, co agent **wie** (np. „pole jest bezpieczne”, „tu na pewno jest jama”),
* **konsekwencje odważne** (prawdziwe w jakimś modelu) — to, co jest **możliwe**.

Na tej podstawie agent wybiera cel (zgodnie z priorytetem: podnieś złoto → wróć z całym złotem → idź na najbliższe pewnie bezpieczne pole → zastrzel ewentualnego wumpusa na polu bez jamy → [opcjonalnie] zaryzykuj → wyjdź z jaskini), a trasę wyznacza przeszukiwaniem wszerz po polach, o których wie, że są bezpieczne.

Pliki: `world.py` (środowisko), `inference.py` (baza wiedzy i wywołania Clingo), `agent.py` (strategia i planowanie), `simulate.py` (pętla gry i statystyki).

## Rozważane warianty

Funkcja `run_game(config, max_actions, risky, agent_knows_gold, verbose)` pozwala porównywać następujące warianty:

1. **Standardowy świat z AIMA** (`config = AIMA_WORLD`) — plansza 4×4 z rys. 7.2: jamy w $(3,1)$, $(3,3)$, $(4,4)$, wumpus w $(1,3)$, złoto w $(2,3)$.
2. **Losowe światy** (`generate_world(width, height, pit_prob, n_wumpus, n_gold, rng=...)`):
   * dowolny rozmiar planszy (także prostokątnej),
   * każde pole poza wejściem jest jamą niezależnie z prawdopodobieństwem `pit_prob` (domyślnie 0.2),
   * **wiele wumpusów** — agent ma tyle strzał, ilu jest wumpusów; wumpus może stać na jamie,
   * **wiele sztuk złota** — złoto leży tylko na polach bez jam.

   Losowy świat może być nierozwiązywalny (np. złoto otoczone jamami albo brak bezpiecznego ruchu już na starcie) — wtedy rozsądny agent po prostu wychodzi z jaskini.
3. **Znana / nieznana liczba sztuk złota** (`agent_knows_gold`) — jeśli agent wie, ile jest złota, wraca od razu po zebraniu wszystkiego; w przeciwnym razie eksploruje, dopóki istnieją pewnie bezpieczne pola.
4. **Agent ostrożny / ryzykujący** (`risky`) — agent ostrożny wchodzi tylko na pola, których bezpieczeństwo **wynika logicznie** z wiedzy; ryzykujący, gdy takich pól brak, wchodzi na najmniej podejrzane pole, które nie jest na pewno śmiertelne (zamiast od razu się wycofać).

Ten sam zestaw wariantów jest dostępny z linii poleceń: `python simulate.py --help`.



## Zadania do wykonania

1. Uruchom symulację i przyjrzyj się, jak agent ostrożny zachowuje się w standardowym świecie AIMA. Zmodyfikuj parametr max_actions, aby sprawdzić co się stanie gdy agentowi skończą się akcje.
2. Jaką maksymalną nagrodę da się uzyskać w tym środowisku (znając rozkład świata)?
3. Dokończ tryb ryzykowny działania agenta (miejsce do uzupełnienia oznaczone komentarzem TODO (RISKY) w pliku agent.py). W trybie ryzykownym agent ma wchodzić na najbliższe pole, na którym może potencjalnie znajdować się wumpus lub jama.
4. Napisz kod rozgrywający po 100 losowych gier (losowy świat) agenta ostrożnego i ryzykownego i obliczający średnią wartość nagrody. Który agent radzi sobie lepiej?
5. Dodaj wariant "zliczający" do trybu ryzykownego: agent estymuje prawdopodobieństwa wystąpienia ryzyka za pomocą metody danger_probabilities i wybiera pole, dla którego ryzyko jest najniższe. Powtórz eksperyment z losowymi światami i porównaj skuteczność agentów.

Poniższy kod uruchamia symulację w świecie AIMA z agentem ostrożnym.

In [1]:
%pip install clingo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations

import argparse
import random
import time
from dataclasses import dataclass

from agent import Agent
from world import AIMA_WORLD, COUNTING_WORLD, World, WorldConfig, generate_world
from simulate import run_game

#config = AIMA_WORLD
config = COUNTING_WORLD
verbose = True
max_actions = 100
risky = False
unknown_gold = False

run_game(config, max_actions, risky, not unknown_gold, verbose)

True world (P pit, W/w live/dead wumpus, G gold):
...  ...  ...  ... 
.W.  ...  ...  ... 
...  P..  ...  ... 
...> ...  ...  ..G 

Agent knowledge description:

        Each cell has 4 characters:
        * hunter position: one of "^", ">", "v", "<" or " " if the hunter is somewhere else
        * general status: "." -> visited, "s" -> provably safe, "?" unknown
        * pit status: "P" -> certain, "p" -> possible, "b" -> breeze felt here, " " -> away from frontier
        * wumpus status: "W" -> certainly alive, "X" -> certainly dead, "w" -> possibly alive, "~" -> stench felt here, " " -> away from frontier
        

--- goal: explore (2, 1)
 ?  | ?  | ?  | ?  
 ?  | ?  | ?  | ?  
 s  | ?  | ?  | ?  
 .  |>s  | ?  | ?  
   1. forward  -> (2, 1) breeze
--- goal: explore (1, 2)
 ?  | ?  | ?  | ?  
 ?  | ?  | ?  | ?  
 s  | ?p | ?  | ?  
 .  |^.b | ?p | ?  
   2. left     -> (2, 1) breeze
   3. left     -> (2, 1) breeze
   4. forward  -> (1, 1) 
   5. right    -> (1, 1) 
   6. forward  

GameResult(score=-10, actions=10, gold=0, alive=True, climbed=True, seconds=0.0323045999975875)

In [3]:
#1) Test w AIMA_WORLD + zmieniejszenie parametru max_actions z 100 do 20

config = AIMA_WORLD
verbose = True
max_actions = 100
risky = False
unknown_gold = False


run_game(config, max_actions, risky, not unknown_gold, verbose)

True world (P pit, W/w live/dead wumpus, G gold):
...  ...  ...  P.. 
.W.  ..G  P..  ... 
...  ...  ...  ... 
...> ...  P..  ... 

Agent knowledge description:

        Each cell has 4 characters:
        * hunter position: one of "^", ">", "v", "<" or " " if the hunter is somewhere else
        * general status: "." -> visited, "s" -> provably safe, "?" unknown
        * pit status: "P" -> certain, "p" -> possible, "b" -> breeze felt here, " " -> away from frontier
        * wumpus status: "W" -> certainly alive, "X" -> certainly dead, "w" -> possibly alive, "~" -> stench felt here, " " -> away from frontier
        

--- goal: explore (2, 1)
 ?  | ?  | ?  | ?  
 ?  | ?  | ?  | ?  
 s  | ?  | ?  | ?  
 .  |>s  | ?  | ?  
   1. forward  -> (2, 1) breeze
--- goal: explore (1, 2)
 ?  | ?  | ?  | ?  
 ?  | ?  | ?  | ?  
 s  | ?p | ?  | ?  
 .  |^.b | ?p | ?  
   2. left     -> (2, 1) breeze
   3. left     -> (2, 1) breeze
   4. forward  -> (1, 1) 
   5. right    -> (1, 1) 
   6. forward  

GameResult(score=978, actions=22, gold=1, alive=True, climbed=True, seconds=0.054831799992825836)

| Limit akcji | Wykonane akcje | Podniesione złoto | Agent wyszedł z jaskini | Punkty |
|---:|---:|---:|:---:|---:|
| 20 | 20 | 1 | Nie | −20 |
| 21 | 21 | 1 | Nie | −21 |
| 22 | 22 | 1 | Tak | 978 |
| 100 | 22 | 1 | Tak | 978 |

### 1. Co się stanie gdy agentowi skończą się akcje ?

ODP: 
"Hunter ran out of actions with 1 gold after 20 actions."

Agent ostrożny poznaje świat na podstawie obserwacji i wchodzi tylko na pola, których bezpieczeństwo potrafi uzasadnić logicznie. Początkowo odwiedza pola (2,1) i (1,2). Powiew i smród pozwalają mu ograniczać możliwe położenia zagrożeń. Następnie eksploruje kolejne bezpieczne pola, dociera do złota na (2,3) i podnosi je w 15. akcji. Ponieważ wie, że w jaskini znajduje się jedna sztuka złota, rozpoczyna powrót do wejścia.

Przy limicie max_actions=100 agent kończy grę po 22 akcjach: przeżywa, wynosi jedną sztukę złota i uzyskuje 978 punktów, czyli 1000 − 22. Nie oddaje żadnego strzału.

Po zmniejszeniu limitu do 20 symulacja zatrzymuje się podczas powrotu. Agent nadal żyje i niesie złoto, ale znajduje się na (2,1). Wynik wynosi −20, ponieważ premia za złoto jest przyznawana dopiero po wyjściu z jaskini.

Przy limicie 21 agent dociera do wejścia (1,1), ale nie zdąża wykonać akcji climb, dlatego otrzymuje −21 punktów. Dopiero limit 22 pozwala zakończyć wyprawę sukcesem. Wyczerpanie limitu nie oznacza śmierci — powoduje zatrzymanie symulacji.

### 2. Jaką maksymalną nagrodę da się uzyskać znając rozkład świata ? 

ODP:
Maksymalna nagroda w standardowym świecie AIMA wynosi 988 punktów.
Znając rozmieszczenie jam, wumpusa i złota, można od razu wybrać bezpieczną trasę:
(1,1) → (2,1) → (2,2) → (2,3)
Na polu (2,3) należy podnieść złoto, a następnie wrócić:
(2,3) → (2,2) → (2,1) → (1,1)

In [4]:
# 3) Tryb risky

config = AIMA_WORLD
verbose = True
max_actions = 100
risky = True
unknown_gold = False


run_game(config, max_actions, risky, not unknown_gold, verbose)

True world (P pit, W/w live/dead wumpus, G gold):
...  ...  ...  P.. 
.W.  ..G  P..  ... 
...  ...  ...  ... 
...> ...  P..  ... 

Agent knowledge description:

        Each cell has 4 characters:
        * hunter position: one of "^", ">", "v", "<" or " " if the hunter is somewhere else
        * general status: "." -> visited, "s" -> provably safe, "?" unknown
        * pit status: "P" -> certain, "p" -> possible, "b" -> breeze felt here, " " -> away from frontier
        * wumpus status: "W" -> certainly alive, "X" -> certainly dead, "w" -> possibly alive, "~" -> stench felt here, " " -> away from frontier
        

--- goal: explore (2, 1)
 ?  | ?  | ?  | ?  
 ?  | ?  | ?  | ?  
 s  | ?  | ?  | ?  
 .  |>s  | ?  | ?  
   1. forward  -> (2, 1) breeze
--- goal: explore (1, 2)
 ?  | ?  | ?  | ?  
 ?  | ?  | ?  | ?  
 s  | ?p | ?  | ?  
 .  |^.b | ?p | ?  
   2. left     -> (2, 1) breeze
   3. left     -> (2, 1) breeze
   4. forward  -> (1, 1) 
   5. right    -> (1, 1) 
   6. forward  

GameResult(score=978, actions=22, gold=1, alive=True, climbed=True, seconds=0.04026680000242777)

### 4. Napisz kod rozgrywający po 100 losowych gier (losowy świat) agenta ostrożnego i ryzykownego i obliczający średnią wartość nagrody. Który agent radzi sobie lepiej?

Porównujemy oba tryby na tych samych 100 losowych planszach 4×4: prawdopodobieństwo jamy 0,2, jeden wumpus i jedna sztuka złota. Stałe ziarno losowości pozwala powtórzyć eksperyment. Agent zna liczbę sztuk złota; limit wynosi 1000 akcji na grę. Nie odrzucamy trudnych ani nierozwiązywalnych plansz.

Główną miarą jest średnia nagroda. Dodatkowo porównujemy przeżywalność, wyniesienie złota i liczbę akcji. Wyłączamy szczegółowe logi (`verbose=False`). Po zmianach w `agent.py` należy zrestartować kernel, aby używać aktualnej implementacji.

In [5]:
import random
from statistics import mean
from IPython.display import Markdown, display
from world import generate_world
from simulate import run_game

N_GAMES = 100
SEED = 42
MAX_ACTIONS = 1000
rng = random.Random(SEED)
worlds = [
    generate_world(width=4, height=4, pit_prob=0.2, n_wumpus=1, n_gold=1, rng=rng)
    for _ in range(N_GAMES)
]
results = {False: [], True: []}
for config in worlds:
    for risky_mode in results:
        results[risky_mode].append(run_game(
            config, max_actions=MAX_ACTIONS, risky=risky_mode,
            agent_knows_gold=True, verbose=False,
        ))

summary = {}
for risky_mode, games in results.items():
    summary[risky_mode] = {
        'reward': mean(game.score for game in games),
        'survival': 100 * mean(game.alive for game in games),
        'success': 100 * mean(
            game.alive and game.climbed and game.gold > 0 for game in games
        ),
        'actions': mean(game.actions for game in games),
        'timeouts': sum(game.alive and not game.climbed for game in games),
    }

table = [
    '| Tryb | Liczba gier | Średnia nagroda | Przeżywalność | Wyniesienie złota | Średnia liczba akcji | Limit akcji |',
    '|---|---:|---:|---:|---:|---:|---:|',
]
for risky_mode, label in [(False, 'Ostrożny'), (True, 'Ryzykowny')]:
    s = summary[risky_mode]
    table.append(
        f"| {label} | {N_GAMES} | {s['reward']:.2f} | {s['survival']:.0f}% "
        f"| {s['success']:.0f}% | {s['actions']:.2f} | {s['timeouts']} |"
    )
display(Markdown('\n'.join(table)))

| Tryb | Liczba gier | Średnia nagroda | Przeżywalność | Wyniesienie złota | Średnia liczba akcji | Limit akcji |
|---|---:|---:|---:|---:|---:|---:|
| Ostrożny | 100 | 427.62 | 100% | 44% | 11.08 | 0 |
| Ryzykowny | 100 | 223.93 | 62% | 62% | 14.17 | 0 |

In [6]:
differences = [risky.score - cautious.score
               for cautious, risky in zip(results[False], results[True])]
risky_wins = sum(d > 0 for d in differences)
cautious_wins = sum(d < 0 for d in differences)
ties = sum(d == 0 for d in differences)
delta = mean(differences)
if delta > 0:
    verdict = 'Wyższą średnią nagrodę uzyskał agent ryzykowny.'
elif delta < 0:
    verdict = 'Wyższą średnią nagrodę uzyskał agent ostrożny.'
else:
    verdict = 'Oba tryby uzyskały taką samą średnią nagrodę.'

display(Markdown(
    f"**Wnioski dla {N_GAMES} plansz (ziarno {SEED}):** {verdict} "
    f"Różnica średnich (ryzykowny − ostrożny) wyniosła **{delta:+.2f} punktów**. "
    f"W porównaniu parami ryzykowny miał wyższą nagrodę na {risky_wins} planszach, "
    f"ostrożny na {cautious_wins}, a na {ties} wystąpił remis.\n\n"
    f"Agent ostrożny wyniósł złoto w {summary[False]['success']:.0f}% gier "
    f"i przeżył {summary[False]['survival']:.0f}%, a ryzykowny odpowiednio "
    f"w {summary[True]['success']:.0f}% i {summary[True]['survival']:.0f}%. "
    "Ryzyko pozwala kontynuować eksplorację po wyczerpaniu bezpiecznych ruchów, "
    "ale może prowadzić do śmierci, za którą odejmowane jest 1000 punktów. "
    "Wynik dotyczy tej próby i tych parametrów; nie dowodzi przewagi jednego trybu "
    "dla wszystkich rozkładów świata."
))

**Wnioski dla 100 plansz (ziarno 42):** Wyższą średnią nagrodę uzyskał agent ostrożny. Różnica średnich (ryzykowny − ostrożny) wyniosła **-203.69 punktów**. W porównaniu parami ryzykowny miał wyższą nagrodę na 18 planszach, ostrożny na 38, a na 44 wystąpił remis.

Agent ostrożny wyniósł złoto w 44% gier i przeżył 100%, a ryzykowny odpowiednio w 62% i 62%. Ryzyko pozwala kontynuować eksplorację po wyczerpaniu bezpiecznych ruchów, ale może prowadzić do śmierci, za którą odejmowane jest 1000 punktów. Wynik dotyczy tej próby i tych parametrów; nie dowodzi przewagi jednego trybu dla wszystkich rozkładów świata.

### 5. Wariant zliczający — wybór pola o najmniejszym prawdopodobieństwie zagrożenia

Parametr `counting=True` w `Agent` i `run_game` włącza wariant ryzykowny, który wywołuje `danger_probabilities(pit_prob)` dopiero po wyczerpaniu bezpiecznych ruchów i możliwości zabezpieczenia pola strzałem. Metoda wylicza ważone modele zgodne z obserwacjami i zwraca prawdopodobieństwo jamy **lub żywego wumpusa**. Agent pomija pola na pewno śmiertelne i wybiera najmniejsze ryzyko; przy remisie wybiera najkrótszą trasę przez pola odwiedzone do wybranego pola pogranicza.

Powtarzamy eksperyment z punktu 4 dla trzech trybów na tych samych 100 planszach: 4×4, jeden wumpus, jedna sztuka złota, `pit_prob=0.2`, ziarno 42 i limit 1000 akcji. Agent zliczający zna prawdopodobieństwo jamy użyte przez generator, ale nie zna rozmieszczenia zagrożeń. Pełne zliczanie modeli może być kosztowne, więc mierzymy również średni czas gry (`verbose=False`). Wniosek dotyczy tej próby, a nie wszystkich możliwych światów.


In [1]:
import random
from statistics import mean
from IPython.display import Markdown, display
from world import generate_world
from simulate import run_game

N_GAMES_5 = 100
SEED_5 = 42
PIT_PROB_5 = 0.2
MAX_ACTIONS_5 = 1000
comparison_rng = random.Random(SEED_5)
comparison_worlds = [
    generate_world(width=4, height=4, pit_prob=PIT_PROB_5,
                   n_wumpus=1, n_gold=1, rng=comparison_rng)
    for _ in range(N_GAMES_5)
]
comparison_modes = {
    'Ostrożny': {'risky': False},
    'Ryzykowny': {'risky': True},
    'Zliczający': {'counting': True},
}
comparison_results = {label: [] for label in comparison_modes}
for config in comparison_worlds:
    for label, options in comparison_modes.items():
        comparison_results[label].append(run_game(
            config, max_actions=MAX_ACTIONS_5, agent_knows_gold=True,
            verbose=False, pit_prob=PIT_PROB_5, **options,
        ))

comparison_summary = {}
for label, games in comparison_results.items():
    comparison_summary[label] = {
        'reward': mean(game.score for game in games),
        'survival': 100 * mean(game.alive for game in games),
        'success': 100 * mean(
            game.alive and game.climbed and game.gold > 0 for game in games
        ),
        'actions': mean(game.actions for game in games),
        'seconds': mean(game.seconds for game in games),
        'timeouts': sum(game.alive and not game.climbed for game in games),
    }

comparison_table = [
    '| Tryb | Gry | Średnia nagroda | Przeżywalność | Wyniesienie złota | Średnie akcje | Średni czas [s] | Gry z wyczerpanym limitem |',
    '|---|---:|---:|---:|---:|---:|---:|---:|',
]
for label, s in comparison_summary.items():
    comparison_table.append(
        f"| {label} | {N_GAMES_5} | {s['reward']:.2f} | {s['survival']:.0f}% "
        f"| {s['success']:.0f}% | {s['actions']:.2f} | {s['seconds']:.4f} | {s['timeouts']} |"
    )
display(Markdown('\n'.join(comparison_table)))


| Tryb | Gry | Średnia nagroda | Przeżywalność | Wyniesienie złota | Średnie akcje | Średni czas [s] | Gry z wyczerpanym limitem |
|---|---:|---:|---:|---:|---:|---:|---:|
| Ostrożny | 100 | 427.62 | 100% | 44% | 11.08 | 0.0211 | 0 |
| Ryzykowny | 100 | 223.93 | 62% | 62% | 14.17 | 0.0283 | 0 |
| Zliczający | 100 | 243.87 | 63% | 63% | 14.33 | 0.0329 | 0 |

In [2]:
best_reward_5 = max(s['reward'] for s in comparison_summary.values())
best_modes_5 = [label for label, s in comparison_summary.items()
                if s['reward'] == best_reward_5]
counted_5 = comparison_summary['Zliczający']
risky_5 = comparison_summary['Ryzykowny']
cautious_5 = comparison_summary['Ostrożny']
paired_5 = [counted.score - risky.score for counted, risky in zip(
    comparison_results['Zliczający'], comparison_results['Ryzykowny']
)]
display(Markdown(
    f"**Wnioski dla {N_GAMES_5} plansz (ziarno {SEED_5}):** Najwyższą średnią nagrodę "
    f"(**{best_reward_5:.2f}**) uzyskał tryb: **{', '.join(best_modes_5)}**. "
    f"Wariant zliczający uzyskał {counted_5['reward']:.2f} punktów; różnica względem "
    f"zwykłego ryzykownego wyniosła **{counted_5['reward'] - risky_5['reward']:+.2f}**, "
    f"a względem ostrożnego **{counted_5['reward'] - cautious_5['reward']:+.2f}**.\n\n"
    f"Zliczający wyniósł złoto w {counted_5['success']:.0f}% gier i przeżył "
    f"{counted_5['survival']:.0f}% gier. Dla zwykłego ryzykownego było to odpowiednio "
    f"{risky_5['success']:.0f}% i {risky_5['survival']:.0f}%. W porównaniu tych dwóch "
    f"trybów na tych samych planszach zliczający uzyskał wyższą nagrodę "
    f"w {sum(d > 0 for d in paired_5)} grach, niższą w {sum(d < 0 for d in paired_5)}, "
    f"a w {sum(d == 0 for d in paired_5)} był remis.\n\n"
    f"Średni czas gry wyniósł {counted_5['seconds']:.4f} s dla zliczającego "
    f"i {risky_5['seconds']:.4f} s dla zwykłego ryzykownego. Zliczanie uwzględnia "
    "prawdopodobieństwa zagrożeń i wymaga wyliczania modeli. Wybór pola o najmniejszym "
    "bieżącym ryzyku nie gwarantuje najwyższej nagrody całej wyprawy: agent nadal może "
    "zginąć, a kara za śmierć wynosi 1000 punktów. Wyniki dotyczą podanych parametrów "
    "i jednej próby 100 plansz."
))


**Wnioski dla 100 plansz (ziarno 42):** Najwyższą średnią nagrodę (**427.62**) uzyskał tryb: **Ostrożny**. Wariant zliczający uzyskał 243.87 punktów; różnica względem zwykłego ryzykownego wyniosła **+19.94**, a względem ostrożnego **-183.75**.

Zliczający wyniósł złoto w 63% gier i przeżył 63% gier. Dla zwykłego ryzykownego było to odpowiednio 62% i 62%. W porównaniu tych dwóch trybów na tych samych planszach zliczający uzyskał wyższą nagrodę w 2 grach, niższą w 1, a w 97 był remis.

Średni czas gry wyniósł 0.0329 s dla zliczającego i 0.0283 s dla zwykłego ryzykownego. Zliczanie uwzględnia prawdopodobieństwa zagrożeń i wymaga wyliczania modeli. Wybór pola o najmniejszym bieżącym ryzyku nie gwarantuje najwyższej nagrody całej wyprawy: agent nadal może zginąć, a kara za śmierć wynosi 1000 punktów. Wyniki dotyczą podanych parametrów i jednej próby 100 plansz.